# ST Guitar — Stage 7G-E3-D-R1 Colab Execution

FAIL-CLOSED TEMPLATE. This notebook must be pinned after the R1A execution-harness merge before training.

Scientific boundary: new E3 curriculum Batch01 Teacher-GOLD development data only; 399 decisive rows; Stage 7E forbidden; nested development CV only; no checkpoint retention; no production integration.


In [ ]:
PINNED_EXECUTION_SHA = "__PIN_AFTER_R1A_MERGE__"
assert PINNED_EXECUTION_SHA != "__PIN_AFTER_R1A_MERGE__", "STOP: notebook is not pinned to the approved R1A merge SHA"
assert len(PINNED_EXECUTION_SHA) == 40


In [ ]:
!git clone -q https://github.com/khfy7wpr5p-maker/st-guitar-fingering-training.git
%cd st-guitar-fingering-training
!git checkout -q $PINNED_EXECUTION_SHA
!pip -q install -e .


Upload exactly these sealed external files:
- `ST_Guitar_Stage7G_E3_B_R1_Curriculum_Batch01_400.zip`
- `ST_Guitar_E3_Batch01_choices_400of400.json`

File names are not trusted by themselves; the next cell verifies frozen SHA-256 values and the ZIP's internal audit/manifest hashes.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
# PRE-FIT IDENTITY + PREFLIGHT — NO MODEL FIT IN THIS CELL
from __future__ import annotations
import json, platform, subprocess
from pathlib import Path
import numpy as np
import sklearn

from st_guitar_fingering_training.stage7g_e3_d_execution import (
    load_stage7g_e3_d_rows,
)

repo_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert repo_sha == PINNED_EXECUTION_SHA, (repo_sha, PINNED_EXECUTION_SHA)

package_path = Path("ST_Guitar_Stage7G_E3_B_R1_Curriculum_Batch01_400.zip")
choices_path = Path("ST_Guitar_E3_Batch01_choices_400of400.json")
assert package_path.is_file()
assert choices_path.is_file()

rows, preflight = load_stage7g_e3_d_rows(package_path, choices_path)
assert preflight["status"] == "PREFLIGHT_PASS_STOP_BEFORE_TRAIN"

identity = {
    "repository": "khfy7wpr5p-maker/st-guitar-fingering-training",
    "approved_git_sha": PINNED_EXECUTION_SHA,
    "actual_git_sha": repo_sha,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "protocol": "7G-E3-D",
    "data_scope": "new_teacher_gold_development_not_untouched_validation",
    "stage7e_mounted_or_used": False,
    "checkpoint_retained": False,
    "production_integration": False,
}
print(json.dumps(identity, indent=2))
print(json.dumps(preflight, indent=2))
print("\n===== STOP: PREFLIGHT PASS. INSPECT ABOVE BEFORE RUNNING THE MANUAL TRAIN CELL =====")
PREFLIGHT_READY = True


In [ ]:
# MANUAL TRAIN CELL — RUN THIS CELL ONLY AFTER INSPECTING THE PREFLIGHT ABOVE
assert PREFLIGHT_READY is True
from st_guitar_fingering_training.stage7g_e3_d_execution import stage7g_e3_d_nested_cv_report

report = stage7g_e3_d_nested_cv_report(rows)
print(json.dumps({
    "status": report["status"],
    "event_accuracy": report["aggregate"]["accuracy"],
    "always_open_low_accuracy": report["aggregate"]["always_open_low_accuracy"],
    "event_delta": report["aggregate"]["accuracy_delta_vs_always_open_low"],
    "macro_family_accuracy": report["aggregate"]["macro_family_accuracy"],
    "macro_family_baseline": report["aggregate"]["macro_family_always_open_low_accuracy"],
    "macro_family_delta": report["aggregate"]["macro_family_accuracy_delta_vs_always_open_low"],
    "compact_precision": report["aggregate"]["compact_precision"],
    "compact_recall": report["aggregate"]["compact_recall"],
    "compact_tp": report["aggregate"]["compact_true_positive"],
    "compact_fp": report["aggregate"]["compact_false_positive"],
    "family_wins": report["aggregate"]["family_wins"],
    "family_ties": report["aggregate"]["family_ties"],
    "family_losses": report["aggregate"]["family_losses"],
    "outer_selected_thresholds": [fold["selected_threshold"] for fold in report["validation"]["outer_folds"]],
}, indent=2))


In [ ]:
# AGGREGATE EVIDENCE EXPORT — NO MODEL/CHECKPOINT FILE
from google.colab import files

result = {
    "schema": "st-guitar-stage7g-e3-d-r1-colab-result-v1",
    "protocol": "7G-E3-D",
    "execution_git_sha": PINNED_EXECUTION_SHA,
    "identity": identity,
    "input_hashes": {
        "package_sha256": preflight["package_sha256"],
        "audit_sha256": preflight["audit_sha256"],
        "manifest_sha256": preflight["manifest_sha256"],
        "choices_sha256": preflight["choices_sha256"],
        "feature_list_sha256": preflight["feature_list_sha256"],
    },
    "preflight": preflight,
    "report": report,
    "checkpoint_retained": False,
    "production_integration": False,
    "stage7e_used": False,
}
output = Path("ST_Guitar_Stage7G_E3_D_R1_result.json")
output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
files.download(str(output))
